# 7043SCN — Generative AI and Reinforcement Learning
## Coursework: Chef's Hat GYM

| | |
|---|---|
| **Student ID** | 16743515 |
| **Variant** | `16743515 mod 7 = 5` → **Algorithm Comparison Variant** |
| **Task** | Implement and compare DQN and PPO on the Chef's Hat competitive card game |
| **Deadline** | Monday 02/03/2026 at 6pm UK time |

---
### Note on Environment
This notebook uses `chefshat_simulator.py` — a faithful built-in implementation of
the Chef's Hat card game that mirrors the ChefsHatGYM API exactly.
Important Note: Figures are saved in the notebook and are given seperately in the repository for evaluation.No external package installation is required beyond PyTorch.

In [1]:

import subprocess, sys

def pip_install(pkg, label=None):
    label = label or pkg
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True, text=True)
    print('OK:' if r.returncode == 0 else 'FAILED:', label)
    if r.returncode != 0:
        print(r.stderr[-300:])

pip_install('torch',      'PyTorch')
pip_install('numpy',      'NumPy')
pip_install('matplotlib', 'Matplotlib')
pip_install('scipy',      'SciPy')

print('\nAll done.')
print('If PyTorch failed, open Anaconda Prompt and run:')
print('  conda install pytorch cpuonly -c pytorch')

OK: PyTorch
OK: NumPy
OK: Matplotlib
OK: SciPy

All done.
If PyTorch failed, open Anaconda Prompt and run:
  conda install pytorch cpuonly -c pytorch


In [2]:
import os, sys, json, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, os.path.abspath('.'))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

from agents.base_rl_agent import OBSERVATION_SIZE, ACTION_SIZE
from agents.dqn_agent     import DQNAgent
from agents.ppo_agent     import PPOAgent
from training.train       import train_agent, evaluate_agent, ChefsHatEnvironment
from utils.visualisation  import (
    plot_win_rate_comparison, plot_loss_curves, plot_reward_distributions,
    plot_comparison_summary, print_comparison_table, rolling_mean, smooth, PALETTE,
)

os.makedirs('results/dqn',     exist_ok=True)
os.makedirs('results/ppo',     exist_ok=True)
os.makedirs('results/figures', exist_ok=True)

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'Observation size : {OBSERVATION_SIZE}')
print(f'Action size      : {ACTION_SIZE}')

PyTorch 2.10.0+cpu | CUDA: False
Observation size : 405
Action size      : 200


---
## 1. Environment & State/Action Space

Chef's Hat is a competitive turn-based card game for exactly 4 players.

| Property | Detail |
|---|---|
| Players | 4 (fixed) |
| Cards | 11 values × 4 copies = 44 cards, dealt 11 each |
| Objective | First to empty hand wins each round |
| Actions | Play N copies of a value V that beats the table, or PASS |
| Reward | Sparse — based on finishing rank at match end |
| Observation | 405-dim float32 vector (hand + table + scores + round) |
| Action space | 200-dim discrete (masked to legal moves each turn) |

In [3]:
from chefshat_simulator import ChefsHatSimulator, ChefsHatMatchRunner

sim  = ChefsHatSimulator(max_rounds=5, seed=SEED)
obs  = sim.get_observation(0)
mask = sim.get_action_mask(0)

print(f'Observation shape : {obs.shape}')
print(f'Action mask shape : {mask.shape}')
print(f'Legal actions     : {int(mask.sum())}')
print(f'Player 0 hand     : {sorted(sim.hands[0])}')
print(f'Reward structure  : rank-1=+1.0, rank-2=+0.33, rank-3=-0.33, rank-4=-1.0')

Observation shape : (405,)
Action mask shape : (200,)
Legal actions     : 12
Player 0 hand     : [1, 2, 2, 3, 4, 7, 7, 9, 10, 10, 11]
Reward structure  : rank-1=+1.0, rank-2=+0.33, rank-3=-0.33, rank-4=-1.0


---
## 3. DQN — Training

In [4]:
DQN_CONFIG = dict(
    name='DQN', obs_size=OBSERVATION_SIZE, action_size=ACTION_SIZE,
    lr=1e-4, gamma=0.99,
    epsilon_start=1.0, epsilon_end=0.05, epsilon_decay_steps=50_000,
    batch_size=64, replay_capacity=50_000,
    target_update_freq=500, min_replay_size=1_000,
    log_dir='results/dqn', seed=SEED,
)
for k, v in DQN_CONFIG.items(): print(f'  {k:<25}: {v}')

  name                     : DQN
  obs_size                 : 405
  action_size              : 200
  lr                       : 0.0001
  gamma                    : 0.99
  epsilon_start            : 1.0
  epsilon_end              : 0.05
  epsilon_decay_steps      : 50000
  batch_size               : 64
  replay_capacity          : 50000
  target_update_freq       : 500
  min_replay_size          : 1000
  log_dir                  : results/dqn
  seed                     : 42


In [5]:
NUM_TRAINING_MATCHES = 1000  # increase for better results (try 2000+)

dqn_agent = DQNAgent(**DQN_CONFIG)

print(f'Training DQN for {NUM_TRAINING_MATCHES} matches...\n')
dqn_history = train_agent(
    agent=dqn_agent, num_matches=NUM_TRAINING_MATCHES,
    log_interval=50, save_interval=250,
    log_dir='results/dqn', seed=SEED,
)

[DQN] Initialised on cpu. Params: lr=0.0001, γ=0.99, ε=[1.0→0.05], batch=64, target_update=500
Training DQN for 1000 matches...

[DQN]    50/1000 | WinRate=0.280 | Score=13.040 | Reward=-0.040 | Loss=0.0000 | ETA=13.8m
[DQN]   100/1000 | WinRate=0.300 | Score=12.830 | Reward=0.027 | Loss=0.0001 | ETA=13.8m
[DQN]   150/1000 | WinRate=0.220 | Score=13.750 | Reward=-0.133 | Loss=0.0001 | ETA=13.5m
[DQN]   200/1000 | WinRate=0.070 | Score=16.160 | Reward=-0.513 | Loss=0.0001 | ETA=12.9m
[DQN]   250/1000 | WinRate=0.020 | Score=18.410 | Reward=-0.766 | Loss=0.0001 | ETA=12.3m
[DQN] Saved to results/dqn\DQN_ckpt_250.pt
[DQN]   300/1000 | WinRate=0.010 | Score=19.730 | Reward=-0.913 | Loss=0.0000 | ETA=11.6m
[DQN]   350/1000 | WinRate=0.000 | Score=20.920 | Reward=-0.973 | Loss=0.0000 | ETA=10.9m
[DQN]   400/1000 | WinRate=0.000 | Score=21.490 | Reward=-0.980 | Loss=0.0000 | ETA=10.1m
[DQN]   450/1000 | WinRate=0.000 | Score=21.990 | Reward=-0.980 | Loss=0.0000 | ETA=9.3m
[DQN]   500/1000 | W

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(dqn_agent.match_wins))
axes[0].plot(x, rolling_mean(dqn_agent.match_wins, 100), color=PALETTE['DQN'], lw=2)
axes[0].axhline(0.25, color='grey', ls='--', label='Random baseline (25%)')
axes[0].set(title='DQN Win Rate (Rolling 100)', xlabel='Match', ylabel='Win Rate')
axes[0].legend()
if dqn_agent.losses:
    xl = np.arange(len(dqn_agent.losses))
    axes[1].plot(xl, smooth(dqn_agent.losses, 100), color=PALETTE['DQN'], lw=2)
    axes[1].set(title='DQN TD Loss', xlabel='Gradient Steps', ylabel='MSE Loss')
plt.tight_layout()
plt.savefig('results/figures/dqn_curves.png', bbox_inches='tight')
plt.show()
print(f'Win rate (last 100): {np.mean(dqn_agent.match_wins[-100:]):.3f}')
print(f'Epsilon final      : {dqn_agent.epsilon:.4f}')

Win rate (last 100): 0.000
Epsilon final      : 0.0500


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4240\270915847.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. PPO — Training

In [7]:
PPO_CONFIG = dict(
    name='PPO', obs_size=OBSERVATION_SIZE, action_size=ACTION_SIZE,
    lr=3e-4, gamma=0.99, gae_lambda=0.95,
    clip_epsilon=0.2, entropy_coef=0.01, value_loss_coef=0.5,
    update_epochs=4, mini_batch_size=32, max_grad_norm=0.5,
    log_dir='results/ppo', seed=SEED,
)
for k, v in PPO_CONFIG.items(): print(f'  {k:<25}: {v}')

  name                     : PPO
  obs_size                 : 405
  action_size              : 200
  lr                       : 0.0003
  gamma                    : 0.99
  gae_lambda               : 0.95
  clip_epsilon             : 0.2
  entropy_coef             : 0.01
  value_loss_coef          : 0.5
  update_epochs            : 4
  mini_batch_size          : 32
  max_grad_norm            : 0.5
  log_dir                  : results/ppo
  seed                     : 42


In [8]:
ppo_agent = PPOAgent(**PPO_CONFIG)

print(f'Training PPO for {NUM_TRAINING_MATCHES} matches...\n')
ppo_history = train_agent(
    agent=ppo_agent, num_matches=NUM_TRAINING_MATCHES,
    log_interval=50, save_interval=250,
    log_dir='results/ppo', seed=SEED,
)

[PPO] Initialised on cpu. Params: lr=0.0003, γ=0.99, λ=0.95, ε_clip=0.2, entropy=0.01, epochs=4
Training PPO for 1000 matches...

[PPO]    50/1000 | WinRate=0.360 | Score=11.960 | Reward=0.239 | Loss=-0.0101 | ETA=5.6m
[PPO]   100/1000 | WinRate=0.190 | Score=15.150 | Reward=-0.293 | Loss=-0.0107 | ETA=5.2m
[PPO]   150/1000 | WinRate=0.220 | Score=15.350 | Reward=-0.400 | Loss=-0.0066 | ETA=4.9m
[PPO]   200/1000 | WinRate=0.710 | Score=8.810 | Reward=0.513 | Loss=-0.0073 | ETA=4.4m
[PPO]   250/1000 | WinRate=1.000 | Score=4.460 | Reward=1.000 | Loss=-0.0123 | ETA=4.0m
[PPO] Saved to results/ppo\PPO_ckpt_250.pt
[PPO]   300/1000 | WinRate=1.000 | Score=3.240 | Reward=1.000 | Loss=-0.0115 | ETA=3.6m
[PPO]   350/1000 | WinRate=1.000 | Score=2.770 | Reward=1.000 | Loss=-0.0106 | ETA=3.3m
[PPO]   400/1000 | WinRate=1.000 | Score=2.800 | Reward=1.000 | Loss=-0.0079 | ETA=3.0m
[PPO]   450/1000 | WinRate=1.000 | Score=3.070 | Reward=1.000 | Loss=-0.0077 | ETA=2.7m
[PPO]   500/1000 | WinRate=1.0

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(ppo_agent.match_wins))
axes[0].plot(x, rolling_mean(ppo_agent.match_wins, 100), color=PALETTE['PPO'], lw=2)
axes[0].axhline(0.25, color='grey', ls='--', label='Random baseline (25%)')
axes[0].set(title='PPO Win Rate (Rolling 100)', xlabel='Match', ylabel='Win Rate')
axes[0].legend()
if ppo_agent.losses:
    xl = np.arange(len(ppo_agent.losses))
    axes[1].plot(xl, smooth(ppo_agent.losses, 30), color=PALETTE['PPO'], lw=2)
    axes[1].set(title='PPO Combined Loss', xlabel='Match Updates', ylabel='Loss')
plt.tight_layout()
plt.savefig('results/figures/ppo_curves.png', bbox_inches='tight')
plt.show()
print(f'Win rate (last 100): {np.mean(ppo_agent.match_wins[-100:]):.3f}')

Win rate (last 100): 1.000


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4240\3684459003.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Comparative Evaluation

In [10]:
EVAL_MATCHES = 200

print(f'Evaluating DQN over {EVAL_MATCHES} matches (greedy, no training)...')
dqn_eval = evaluate_agent(dqn_agent, num_matches=EVAL_MATCHES, seed=99, log_dir='results/dqn')

print(f'\nEvaluating PPO over {EVAL_MATCHES} matches...')
ppo_eval = evaluate_agent(ppo_agent, num_matches=EVAL_MATCHES, seed=99, log_dir='results/ppo')

print_comparison_table(dqn_eval, ppo_eval)

Evaluating DQN over 200 matches (greedy, no training)...

[EVAL | DQN] 200 matches:
  win_rate        = 0.0000
  avg_rank        = 3.9650
  avg_score       = 20.8050
  std_rank        = 0.1838
  std_score       = 2.7923

Evaluating PPO over 200 matches...

[EVAL | PPO] 200 matches:
  win_rate        = 0.9900
  avg_rank        = 1.0100
  avg_score       = 4.1100
  std_rank        = 0.0995
  std_score       = 2.3828

Metric                           DQN        PPO     Better
win_rate                      0.0000     0.9900        PPO
avg_rank                      3.9650     1.0100        PPO
avg_score                    20.8050     4.1100        PPO
std_rank                      0.1838     0.0995        PPO
std_score                     2.7923     2.3828        PPO



In [11]:
fig = plot_win_rate_comparison(
    dqn_agent.match_wins, ppo_agent.match_wins, window=100,
    save_path='results/figures/win_rate_comparison.png')
plt.show()

fig = plot_loss_curves(
    dqn_agent.losses, ppo_agent.losses,
    save_path='results/figures/loss_comparison.png')
plt.show()

fig = plot_reward_distributions(
    dqn_agent.training_rewards, ppo_agent.training_rewards,
    save_path='results/figures/reward_distributions.png')
plt.show()

fig = plot_comparison_summary(
    dqn_history, ppo_history,
    save_path='results/figures/summary.png')
plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4240\2183601309.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4240\2183601309.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4240\2183601309.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4240\2183601309.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. Hyperparameter Sensitivity Analysis

In [12]:
HP_MATCHES = 300

# DQN: fast vs slow epsilon decay
dqn_hp = {}
for name, decay in [('fast_decay', 20_000), ('slow_decay', 100_000)]:
    a = DQNAgent(name=f'DQN_{name}', epsilon_decay_steps=decay,
                 log_dir=f'results/hp/{name}', seed=SEED)
    train_agent(a, num_matches=HP_MATCHES, log_interval=100,
                save_interval=HP_MATCHES+1, log_dir=f'results/hp/{name}')
    dqn_hp[name] = a.match_wins
    print(f'  DQN {name}: win_rate={np.mean(a.match_wins[-100:]):.3f}')

[DQN] Initialised on cpu. Params: lr=0.0001, γ=0.99, ε=[1.0→0.05], batch=64, target_update=500
[DQN_fast_decay]   100/300 | WinRate=0.140 | Score=15.220 | Reward=-0.333 | Loss=0.0000 | ETA=2.7m
[DQN_fast_decay]   200/300 | WinRate=0.020 | Score=19.750 | Reward=-0.927 | Loss=0.0000 | ETA=1.4m
[DQN_fast_decay]   300/300 | WinRate=0.000 | Score=19.270 | Reward=-0.953 | Loss=0.0000 | ETA=0.0m
[DQN] Saved to results/hp/fast_decay\DQN_fast_decay_final.pt

[DQN_fast_decay] Training complete.
  DQN fast_decay: win_rate=0.000
[DQN] Initialised on cpu. Params: lr=0.0001, γ=0.99, ε=[1.0→0.05], batch=64, target_update=500
[DQN_slow_decay]   100/300 | WinRate=0.340 | Score=12.530 | Reward=0.140 | Loss=0.0002 | ETA=2.8m
[DQN_slow_decay]   200/300 | WinRate=0.360 | Score=12.190 | Reward=0.219 | Loss=0.0007 | ETA=1.4m
[DQN_slow_decay]   300/300 | WinRate=0.270 | Score=13.040 | Reward=0.013 | Loss=0.0006 | ETA=0.0m
[DQN] Saved to results/hp/slow_decay\DQN_slow_decay_final.pt

[DQN_slow_decay] Training 

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# DQN epsilon sensitivity
for name, wins, c in [('fast_decay','#1f77b4',None),('slow_decay','#aec7e8',None)]:
    if name in dqn_hp:
        axes[0].plot(rolling_mean(dqn_hp[name], 50), label=name, lw=2)
axes[0].axhline(0.25, color='grey', ls='--', label='Baseline')
axes[0].set(title='DQN: Epsilon Decay Sensitivity', xlabel='Match', ylabel='Win Rate')
axes[0].legend()

# PPO clip sensitivity
ppo_hp = {}
for clip, color in [(0.1, '#d62728'), (0.3, '#f7b6d2')]:
    name = f'clip_{clip}'
    a = PPOAgent(name=f'PPO_{name}', clip_epsilon=clip,
                 log_dir=f'results/hp/{name}', seed=SEED)
    train_agent(a, num_matches=HP_MATCHES, log_interval=100,
                save_interval=HP_MATCHES+1, log_dir=f'results/hp/{name}')
    ppo_hp[name] = a.match_wins
    axes[1].plot(rolling_mean(a.match_wins, 50), label=f'clip={clip}', lw=2, color=color)
    print(f'  PPO clip={clip}: win_rate={np.mean(a.match_wins[-100:]):.3f}')

axes[1].axhline(0.25, color='grey', ls='--', label='Baseline')
axes[1].set(title='PPO: Clip Epsilon Sensitivity', xlabel='Match', ylabel='Win Rate')
axes[1].legend()
plt.tight_layout()
plt.savefig('results/figures/hp_sensitivity.png', bbox_inches='tight')
plt.show()

[PPO] Initialised on cpu. Params: lr=0.0003, γ=0.99, λ=0.95, ε_clip=0.1, entropy=0.01, epochs=4
[PPO_clip_0.1]   100/300 | WinRate=0.230 | Score=13.200 | Reward=-0.067 | Loss=-0.0036 | ETA=1.1m
[PPO_clip_0.1]   200/300 | WinRate=0.010 | Score=20.030 | Reward=-0.913 | Loss=-0.0125 | ETA=0.6m
[PPO_clip_0.1]   300/300 | WinRate=0.000 | Score=20.030 | Reward=-0.886 | Loss=-0.0134 | ETA=0.0m
[PPO] Saved to results/hp/clip_0.1\PPO_clip_0.1_final.pt

[PPO_clip_0.1] Training complete.
  PPO clip=0.1: win_rate=0.000
[PPO] Initialised on cpu. Params: lr=0.0003, γ=0.99, λ=0.95, ε_clip=0.3, entropy=0.01, epochs=4
[PPO_clip_0.3]   100/300 | WinRate=0.590 | Score=10.160 | Reward=0.340 | Loss=-0.0070 | ETA=1.0m
[PPO_clip_0.3]   200/300 | WinRate=0.990 | Score=4.370 | Reward=0.993 | Loss=-0.0110 | ETA=0.5m
[PPO_clip_0.3]   300/300 | WinRate=0.990 | Score=3.190 | Reward=0.993 | Loss=-0.0053 | ETA=0.0m
[PPO] Saved to results/hp/clip_0.3\PPO_clip_0.3_final.pt

[PPO_clip_0.3] Training complete.
  PPO clip

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4240\1678943404.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
summary = {
    'student_id': '16743515', 'variant': 5,
    'training_matches': NUM_TRAINING_MATCHES,
    'eval_results': {'DQN': dqn_eval, 'PPO': ppo_eval},
    'dqn_config': {k: str(v) for k, v in DQN_CONFIG.items()},
    'ppo_config': {k: str(v) for k, v in PPO_CONFIG.items()},
}
with open('results/final_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Results saved to results/final_results.json')
print('Figures saved to results/figures/')
print('Models saved to results/dqn/ and results/ppo/')
print('Notebook complete.')

Results saved to results/final_results.json
Figures saved to results/figures/
Models saved to results/dqn/ and results/ppo/
Notebook complete.
